# NB125 — The Unified Mass Architecture

## Purpose

This notebook is not hunting for identities. It is trying to understand the truth.

NB124 found that $m_\tau/m_\mu = C_0^{x_3} \times p_3/p_4$ matches the SM to $-0.016\%$. 
But finding a formula that works is not the same as understanding *why* it works. 
The factor $5/7$ could be:

1. **A real dissipation transfer** — the overdamped R₃ level genuinely attenuates the R₄ signal by $\sqrt{\gamma_2/\gamma_3}$
2. **A numerical coincidence** — some other mechanism produces the correction and $5/7$ just happens to be close
3. **Partially correct** — $5/7$ captures the leading effect but misses structure at the $0.016\%$ level

This notebook interrogates the mechanism directly. We decompose the cascade ODE,
measure the actual signal transfer, test for degeneracies, derive the correction
from the Green's function, and run a falsification battery.

## Status entering
- 270 identities, 0 free parameters
- NB124: $m_\tau/m_\mu = C_0^{x_3} \times 5/7 = 16.814$ ($-0.016\%$, PASS)
- Complete quark hierarchy and $m_\mu/m_e$ solved
- Open question: **Is the $p_3/p_4$ correction derived or assumed?**

In [1]:
# -- S0: Setup and Load Cascade Infrastructure --
import sys, numpy as np, time
from pathlib import Path
from fractions import Fraction
from math import gcd

ROOT = Path.cwd().parent
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               PRIMES, P, PHI, GROUP_EXPONENT, PRIMORIALS,
                               X4, X3, X2, LAM7, X4_LEP,
                               CP_PAIRS, SM_TARGETS, PHYSICAL_CROSSINGS,
                               ACTIVE_PRIMES)
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

p1, p2, p3, p4 = SA.primes
P4 = SA.P           # 210
PHI_P4 = SA.PHI     # 48
LAM_P4 = SA.group_exponent  # 12

# Dissipation eigenvalues (NB115)
gamma = {k: int(pk)**2 for k, pk in enumerate(SA.primes)}

# SM targets
M_TAU_OVER_M_MU = 16.817   # PDG 2024
M_MU_OVER_M_E   = 206.768  # PDG 2024
M_TAU_OVER_M_E  = M_TAU_OVER_M_MU * M_MU_OVER_M_E

sys0 = SolenoidSystem()
all_branches = sys0.all_branches()

print(f'JAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

print()
print('NB125: THE UNIFIED MASS ARCHITECTURE')
print('=' * 65)
print(f'  P4 = {P4}, phi(P4) = {PHI_P4}, lambda(P4) = {LAM_P4}')
print(f'  kappa = epsilon = rho = 1/sqrt(210) = {RHO:.6f}')
print(f'  omega = 2*pi = {OMEGA:.6f}')
print(f'  Exponents:')
print(f'    x3     = lambda(P4)/(2pi) = {LAM_P4}/(2pi) = {X3:.6f}')
print(f'    x4     = phi(P4)/(2pi)    = {PHI_P4}/(2pi) = {X4:.6f}')
print(f'    x4_lep = p4^2/(2pi)       = {p4**2}/(2pi) = {X4_LEP:.6f}')
print(f'    x2     = phi(P3)/(2pi)    = 8/(2pi)     = {X2:.6f}')
print(f'    LAM7   = phi(p4)           = {LAM7}')
print(f'  Dissipation eigenvalues gamma_k = p_k^2: {[gamma[k] for k in range(4)]}')
print(f'  sqrt(gamma2/gamma3) = p3/p4 = {p3}/{p4} = {p3/p4:.6f}')
print(f'  SM targets:')
print(f'    m_tau/m_mu = {M_TAU_OVER_M_MU}')
print(f'    m_mu/m_e   = {M_MU_OVER_M_E}')
print(f'    m_tau/m_e  = {M_TAU_OVER_M_E:.1f}')

JAX device: CPU (1 device(s))
JAX warmup: 2.1s

NB125: THE UNIFIED MASS ARCHITECTURE
  P4 = 210, phi(P4) = 48, lambda(P4) = 12
  kappa = epsilon = rho = 1/sqrt(210) = 0.069007
  omega = 2*pi = 6.283185
  Exponents:
    x3     = lambda(P4)/(2pi) = 12/(2pi) = 1.909859
    x4     = phi(P4)/(2pi)    = 48/(2pi) = 7.639437
    x4_lep = p4^2/(2pi)       = 49/(2pi) = 7.798592
    x2     = phi(P3)/(2pi)    = 8/(2pi)     = 1.273240
    LAM7   = phi(p4)           = 6
  Dissipation eigenvalues gamma_k = p_k^2: [4, 9, 25, 49]
  sqrt(gamma2/gamma3) = p3/p4 = 5/7 = 0.714286
  SM targets:
    m_tau/m_mu = 16.817
    m_mu/m_e   = 206.768
    m_tau/m_e  = 3477.2


## Section 1: Decompose Window-0 R₃ CP Ratio into Transient vs Steady-State

NB104 (#228) established: $R_k(t; \text{br}) = R_k^{ss}(t; j_1,...,j_k) + 2\pi j_{k+1} \cdot e^{-\kappa t}$

This is machine-exact at all 4 levels. For R₃ (level index 2):
- **Transient**: $2\pi j_4 \cdot e^{-\kappa \cdot ci}$ — carries j₄ information, generates ALL CP asymmetry
- **Steady-state**: $R_3^{ss}(ci; j_1, j_2, j_3)$ — depends only on lower-level ICs

If $p_3/p_4$ is a real dissipation effect, the transient-only and steady-state-only CP 
ratios should show structured differences. The transient *generates* the asymmetry; 
the steady-state *receives* it through the filter. Their relative contributions 
should reveal the mechanism.

In [ ]:
# -- S1: Integrate and decompose R3 into transient + steady-state --

T_MAX = 5000
cis = SA.coprime_indices(T_MAX)
t_cross = cis.astype(float)         # Convention A: t = ci
T_integ = float(T_MAX + 1)
WINDOW_SIZE = PHI_P4  # 48

# CRT labels
a3_t, a5_t, a7_t = SA.sector_labels(cis)

print(f'Integrating {len(all_branches)} branches to T={T_MAX}...')
t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_cross, T_integ, backend='jax')
dt = time.time() - t0
print(f'Done in {dt:.1f}s. {len(cis)} crossings.')

# Window-0: first 48 coprime crossings
w0_cis = cis[:WINDOW_SIZE]
w0_a3  = a3_t[:WINDOW_SIZE]
w0_a5  = a5_t[:WINDOW_SIZE]
w0_a7  = a7_t[:WINDOW_SIZE]
branches_list = list(res.keys())

# For each branch, decompose R3 (index 2) into transient + steady-state
# R3(ci; br) = R3_ss(ci; j1,j2,j3) + 2*pi*j4 * exp(-kappa*ci)
# transient_k = 2*pi*j_{k+1} * exp(-kappa*ci) at level k (k=0..3)

R3_full = {}       # branch -> array of R3 values at window-0 crossings
R3_trans = {}      # transient component
R3_ss = {}         # steady-state component

for br in branches_list:
    j1, j2, j3, j4 = br
    r_vals = res[br][:WINDOW_SIZE, :]  # shape (48, 4), columns = R0,R1,R2,R3
    r3 = r_vals[:, 2]                  # R3 = level index 2
    trans = 2 * np.pi * j4 * np.exp(-KAPPA * w0_cis.astype(float))
    R3_full[br] = r3
    R3_trans[br] = trans
    R3_ss[br] = r3 - trans

# Verify decomposition is exact
sample_br = branches_list[42]
recon = R3_trans[sample_br] + R3_ss[sample_br]
max_err = np.max(np.abs(recon - R3_full[sample_br]))
print(f'Decomposition check (branch {sample_br}): max |R3 - (trans+ss)| = {max_err:.2e}')

# Now compute CP ratios from transient-only and ss-only
# CP ratio = sqrt(RMS_g1^2 / RMS_g2^2) where g1,g2 are the CRT sector pair
# For LEPTON: a3=0, a7_g1=1, a7_g2=5

lep_a3, lep_g1, lep_g2 = CP_PAIRS['LEPTON']

# Accumulate sector RMS for transient-only and ss-only at R3 (level 2)
def compute_cp_from_component(component_dict, cis_arr, a3_arr, a7_arr, channel_a3, g1_a7, g2_a7, level=2):
    """Compute CP ratio at given level from a component dict {branch: array}."""
    sum2_g1 = np.zeros(len(cis_arr))
    sum2_g2 = np.zeros(len(cis_arr))
    count_g1 = 0
    count_g2 = 0
    
    for br, vals in component_dict.items():
        # vals is the component at each crossing for this branch
        for i, ci in enumerate(cis_arr):
            a3_ci = a3_arr[i]
            a7_ci = a7_arr[i]
            if a3_ci == channel_a3:
                if a7_ci == g1_a7:
                    sum2_g1[i] += vals[i]**2
                elif a7_ci == g2_a7:
                    sum2_g2[i] += vals[i]**2
    
    rms_g1 = np.sqrt(np.sum(sum2_g1) / max(np.count_nonzero(sum2_g1), 1))
    rms_g2 = np.sqrt(np.sum(sum2_g2) / max(np.count_nonzero(sum2_g2), 1))
    return rms_g1 / rms_g2 if rms_g2 > 0 else np.nan

# This brute-force approach is too slow for 210 branches x 48 crossings.
# Use the vectorized accumulate_sectors approach instead.
# We need to construct "fake" result dicts that hold only the component.

# Construct component result dicts in the format expected by accumulate_sectors
# accumulate_sectors expects: {branch: array of shape (n_crossings, 4)}

def make_result_dict(component_dict, n_crossings, n_levels=4, target_level=2):
    """Wrap a single-level component into the 4-level format."""
    result = {}
    for br, vals in component_dict.items():
        arr = np.zeros((n_crossings, n_levels))
        arr[:, target_level] = vals
        result[br] = arr
    return result

trans_res = make_result_dict(R3_trans, WINDOW_SIZE)
ss_res = make_result_dict(R3_ss, WINDOW_SIZE)
full_res_w0 = {b: res[b][:WINDOW_SIZE, :] for b in branches_list}

# Sector accumulation and CP ratios
sec_full = sys0.accumulate_sectors(full_res_w0, w0_cis, w0_a3, w0_a5, w0_a7)
cp_full  = sys0.cp_pair_ratios(sec_full)

sec_trans = sys0.accumulate_sectors(trans_res, w0_cis, w0_a3, w0_a5, w0_a7)
cp_trans  = sys0.cp_pair_ratios(sec_trans)

sec_ss = sys0.accumulate_sectors(ss_res, w0_cis, w0_a3, w0_a5, w0_a7)
cp_ss  = sys0.cp_pair_ratios(sec_ss)

print()
print('WINDOW-0 R3 CP DECOMPOSITION (LEPTON CHANNEL):')
print('=' * 65)
print(f'  {"Component":<20} {"R3 CP ratio":>12} {"C0^x3":>12} {"C0^x3*5/7":>12} {"vs SM":>10}')
print(f'  {"-"*65}')
for label, cp_dict in [('Full R3', cp_full), ('Transient only', cp_trans), ('Steady-state only', cp_ss)]:
    c0 = cp_dict['LEPTON'][2]
    pred_raw = c0 ** X3
    pred_corr = pred_raw * p3/p4
    dev = (pred_corr / M_TAU_OVER_M_MU - 1) * 100
    print(f'  {label:<20} {c0:>12.6f} {pred_raw:>12.4f} {pred_corr:>12.4f} {dev:>+9.4f}%')

C0_R3_L = cp_full['LEPTON'][2]
print(f'\n  Key: C0(R3, lepton, full) = {C0_R3_L:.8f}')
print(f'  Transient CP = {cp_trans["LEPTON"][2]:.8f}')
print(f'  Steady-state CP = {cp_ss["LEPTON"][2]:.8f}')
print(f'\n  The CP asymmetry is {"ENTIRELY" if abs(cp_ss["LEPTON"][2] - 1.0) < 0.01 else "partially"} in the transient.')

## Section 2: R₄→R₃ Signal Transfer — Branch-by-Branch Amplitude Analysis

If $p_3/p_4$ represents a true amplitude transfer between cascade levels, then for each 
branch the ratio of R₃ to R₄ amplitudes should cluster near $5/7$ or a simple function of it.

For every branch $(j_1, j_2, j_3, j_4)$, we extract R₃ and R₄ at each window-0 crossing 
and compute the amplitude ratios. The distribution of these ratios across all 210 branches 
reveals whether $5/7$ is a universal transfer coefficient or an averaged artifact.

In [ ]:
# -- S2: Branch-by-branch R4/R3 amplitude transfer --
import matplotlib.pyplot as plt

# For each branch, compute RMS of R3 transient and R4 transient at window-0 crossings
# R3_trans = 2*pi*j4 * exp(-kappa*ci)
# R4_trans = 2*pi*j5 — but there is no j5 (only 4 levels)!
# Actually: R4 is the outermost level. R4(ci;br) = R4_ss(ci;j1,j2,j3,j4) + ??? 
# There is no level beyond R4, so R4 has NO transient in the same sense.
# R4_trans interpretation: at R4, the initial condition IS the signal.
# R4(0; br) = 2*pi*j4 for the cascade IC.

# Better approach: compare R3 RMS and R4 RMS at each crossing for each branch.
# The ratio RMS(R3)/RMS(R4) across branches reveals the transfer.

ratios_per_branch = []
r3_rms_list = []
r4_rms_list = []

for br in branches_list:
    r3 = res[br][:WINDOW_SIZE, 2]  # R3
    r4 = res[br][:WINDOW_SIZE, 3]  # R4
    rms3 = np.sqrt(np.mean(r3**2))
    rms4 = np.sqrt(np.mean(r4**2))
    if rms4 > 1e-10:
        ratios_per_branch.append(rms3 / rms4)
    r3_rms_list.append(rms3)
    r4_rms_list.append(rms4)

ratios = np.array(ratios_per_branch)
r3_rms = np.array(r3_rms_list)
r4_rms = np.array(r4_rms_list)

print('R3/R4 AMPLITUDE TRANSFER (all 210 branches, window-0):')
print('=' * 65)
print(f'  Mean(R3_rms / R4_rms) = {np.mean(ratios):.6f}')
print(f'  Median                = {np.median(ratios):.6f}')
print(f'  Std                   = {np.std(ratios):.6f}')
print(f'  Min                   = {np.min(ratios):.6f}')
print(f'  Max                   = {np.max(ratios):.6f}')
print()
print(f'  p3/p4 = {p3/p4:.6f}')
print(f'  Mean/target = {np.mean(ratios)/(p3/p4):.6f}')
print()

# The ratio might not be p3/p4 directly — the amplitude transfer happens
# at the level of the CP ratio, not individual R values.
# Let's check the TRANSIENT amplitudes specifically.
trans_ratios = []
for br in branches_list:
    j1, j2, j3, j4 = br
    # R3 transient = 2*pi*j4 * exp(-kappa*ci)
    # R4 has no simple transient (it's outermost)
    # Instead check: what is the R4 RMS vs the R3 transient RMS?
    r3_t = 2 * np.pi * j4 * np.exp(-KAPPA * w0_cis.astype(float))
    r4   = res[br][:WINDOW_SIZE, 3]
    rms_r3t = np.sqrt(np.mean(r3_t**2))
    rms_r4  = np.sqrt(np.mean(r4**2))
    if rms_r4 > 1e-10:
        trans_ratios.append(rms_r3t / rms_r4)

trans_ratios = np.array(trans_ratios)
print(f'  R3_transient_rms / R4_full_rms:')
print(f'    Mean   = {np.mean(trans_ratios):.6f}')
print(f'    Median = {np.median(trans_ratios):.6f}')
print(f'    Std    = {np.std(trans_ratios):.6f}')

# Histogram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(ratios, bins=30, alpha=0.7, edgecolor='black')
ax1.axvline(p3/p4, color='red', linestyle='--', label=f'p3/p4 = {p3/p4:.4f}')
ax1.axvline(np.mean(ratios), color='blue', linestyle=':', label=f'Mean = {np.mean(ratios):.4f}')
ax1.set_xlabel('R3_rms / R4_rms')
ax1.set_ylabel('Count')
ax1.set_title('Full R3/R4 amplitude ratio')
ax1.legend()

ax2.hist(trans_ratios, bins=30, alpha=0.7, edgecolor='black', color='orange')
ax2.axvline(p3/p4, color='red', linestyle='--', label=f'p3/p4 = {p3/p4:.4f}')
ax2.axvline(np.mean(trans_ratios), color='blue', linestyle=':', label=f'Mean = {np.mean(trans_ratios):.4f}')
ax2.set_xlabel('R3_trans_rms / R4_rms')
ax2.set_ylabel('Count')
ax2.set_title('Transient R3/R4 ratio')
ax2.legend()

plt.tight_layout()
plt.savefig(ROOT / 'output' / 'nb125_amplitude_transfer.png', dpi=150)
plt.show()

print(f'\n  NOTE: The individual branch ratios will NOT be {p3/p4:.4f}.')
print(f'  The correction operates at the SECTOR level (RMS over branches),')
print(f'  not at the individual branch level. This is expected.')